# Assertions — Canonical CRSP Common-Stock Source

This notebook independently validates the compact point-in-time source produced by `notebooks/00_build_common_stock_source.ipynb`.

It is strictly read-only. The tests reconstruct the CRSP ordinary-common-stock rule from the original classification columns rather than trusting the published flags, reconcile the exact retained `PERMNO` set with the immutable RAW panel, verify complete-path row preservation, and authenticate the output manifest.

All large scans use Polars lazy execution with projection pushdown and the streaming engine. File hashing uses a fixed-size buffer.

In [ ]:
# 1. CONFIGURATION
from __future__ import annotations

from datetime import date
from hashlib import sha256
import json
from pathlib import Path

import polars as pl
import pyarrow.parquet as pq

ROOT = Path.cwd().resolve()
if ROOT.name in {'notebooks', 'tests'}:
    ROOT = ROOT.parent

SOURCE = ROOT / 'data' / 'crsp_daily_shumway_delisting_processed.parquet'
OUTPUT = ROOT / 'data' / 'processed' / 'crsp_daily_common_stock_pit_source-1990-2025.parquet'
MANIFEST = ROOT / 'data' / 'processed' / 'crsp_daily_common_stock_pit_source-1990-2025.manifest.json'

START_DATE = date(1990, 1, 1)
END_DATE = date(2025, 12, 31)
INVESTABLE_EXCHANGES = ['N', 'A', 'Q']
COMMON_ISSUER_TYPES = ['ACOR', 'CORP']

VERIFY_OUTPUT_SHA256 = True
VERIFY_SOURCE_SHA256 = False  # enable for the final archival audit; reads the 4 GiB source once

for path in (SOURCE, OUTPUT, MANIFEST):
    assert path.exists(), f'Missing required artifact: {path}'

print(f'Polars : {pl.__version__}')
print(f'Source : {SOURCE}')
print(f'Output : {OUTPUT}')

## 2. Independent metadata and manifest contract

The expected output schema is frozen in this test rather than imported from the production notebook. This prevents a simultaneous accidental change to the producer and its manifest from passing unnoticed.

In [ ]:
ESSENTIAL_COLUMNS = [
    'PERMNO', 'PERMCO', 'CUSIP', 'Ticker', 'DlyCalDt',
    'DlyRet', 'DlyRetx', 'DlyRetI',
    'DlyPrc', 'DlyClose', 'DlyHigh', 'DlyLow', 'DlyOpen',
    'DlyCap', 'DlyVol', 'DlyPrcVol', 'ShrOut',
    'vwretd', 'vwretx', 'ewretd', 'sprtrn',
    'PrimaryExch', 'SICCD', 'NAICS',
    'DelistingDt', 'DelRet', 'DelReasonType', 'delist_category',
]

CLASSIFICATION_COLUMNS = [
    'ConditionalType', 'TradingStatusFlg', 'SecurityActiveFlg',
    'SecurityType', 'SecuritySubType', 'ShareType',
    'USIncFlg', 'IssuerType', 'ShrAdrFlg',
]

FLAG_COLUMNS = [
    'is_common_stock_10_11', 'is_regular_active',
    'is_investable_exchange', 'is_formation_eligible',
    'is_adr_diagnostic', 'is_reit_diagnostic', 'is_fund_diagnostic',
]

EXPECTED_COLUMNS = ESSENTIAL_COLUMNS + CLASSIFICATION_COLUMNS + FLAG_COLUMNS

with MANIFEST.open('r', encoding='utf-8') as handle:
    manifest = json.load(handle)

source_file = pq.ParquetFile(SOURCE)
output_file = pq.ParquetFile(OUTPUT)
source_meta = source_file.metadata
output_meta = output_file.metadata

assert output_file.schema_arrow.names == EXPECTED_COLUMNS
assert manifest['schema_version'] == 'crsp-common-stock-pit-source-v1'
assert manifest['output_columns'] == EXPECTED_COLUMNS
assert manifest['output_path'] == str(OUTPUT.relative_to(ROOT))
assert manifest['output_rows'] == output_meta.num_rows
assert manifest['output_size_bytes'] == OUTPUT.stat().st_size
assert manifest['n_retained_permnos'] == manifest['audit_summary']['permnos']

source_signature = manifest['source_signature']
assert source_signature['path'] == str(SOURCE.relative_to(ROOT))
assert source_signature['size_bytes'] == SOURCE.stat().st_size
assert source_signature['mtime_ns'] == SOURCE.stat().st_mtime_ns
assert source_signature['rows'] == source_meta.num_rows
assert source_signature['row_groups'] == source_meta.num_row_groups

configuration = manifest['configuration']
assert configuration['start_date'] == START_DATE.isoformat()
assert configuration['end_date'] == END_DATE.isoformat()
assert configuration['investable_exchanges'] == INVESTABLE_EXCHANGES
assert configuration['common_stock_mapping'] == {
    'SecurityType': 'EQTY',
    'SecuritySubType': 'COM',
    'ShareType': 'NS',
    'USIncFlg': 'Y',
    'IssuerType': COMMON_ISSUER_TYPES,
}

print('PASS frozen schema')
print('PASS manifest paths, dimensions, and source metadata')
print('PASS declared CRSP common-stock mapping')

## 3. Independent reconstruction of every PIT flag

Each stored flag is recomputed directly from the retained CRSP classification fields. Null classifications are treated as ineligible. The test also verifies that no ADR, REIT, or fund diagnostic can coexist with formation eligibility.

In [ ]:
def normalized_code(column: str) -> pl.Expr:
    return (
        pl.col(column)
        .cast(pl.Utf8)
        .str.strip_chars()
        .str.to_uppercase()
    )


expected_common = (
    (normalized_code('SecurityType') == 'EQTY')
    & (normalized_code('SecuritySubType') == 'COM')
    & (normalized_code('ShareType') == 'NS')
    & (normalized_code('USIncFlg') == 'Y')
    & normalized_code('IssuerType').is_in(COMMON_ISSUER_TYPES)
).fill_null(False)

expected_regular = (
    (normalized_code('ConditionalType') == 'RW')
    & (normalized_code('TradingStatusFlg') == 'A')
).fill_null(False)

expected_exchange = normalized_code('PrimaryExch').is_in(INVESTABLE_EXCHANGES).fill_null(False)
expected_formation = (expected_common & expected_regular & expected_exchange).fill_null(False)

expected_adr = (
    normalized_code('SecurityType').is_in(['ADRT', 'ADR'])
    | normalized_code('ShrAdrFlg').is_in(['Y', 'YES', 'ADR'])
).fill_null(False)

expected_reit = (
    normalized_code('SecuritySubType').is_in(['REIT', 'REITS'])
    | normalized_code('IssuerType').is_in(['REIT', 'REITS'])
).fill_null(False)

expected_fund = (
    normalized_code('SecurityType').is_in(['FUND', 'ETF', 'ETMF'])
    | normalized_code('SecuritySubType').is_in([
        'CEF', 'CETF', 'ETF', 'ETMF', 'FUND', 'MF', 'OEF', 'UIT',
    ])
).fill_null(False)

output_lf = pl.scan_parquet(OUTPUT, low_memory=True, rechunk=False)

flag_audit = output_lf.select(
    pl.len().alias('rows'),
    pl.col('PERMNO').n_unique().alias('permnos'),
    pl.col('DlyCalDt').min().alias('date_min'),
    pl.col('DlyCalDt').max().alias('date_max'),
    *[pl.col(column).null_count().alias(f'{column}_nulls') for column in FLAG_COLUMNS],
    (pl.col('is_common_stock_10_11') != expected_common).fill_null(True).sum().alias('common_mismatch'),
    (pl.col('is_regular_active') != expected_regular).fill_null(True).sum().alias('regular_mismatch'),
    (pl.col('is_investable_exchange') != expected_exchange).fill_null(True).sum().alias('exchange_mismatch'),
    (pl.col('is_formation_eligible') != expected_formation).fill_null(True).sum().alias('formation_mismatch'),
    (pl.col('is_adr_diagnostic') != expected_adr).fill_null(True).sum().alias('adr_mismatch'),
    (pl.col('is_reit_diagnostic') != expected_reit).fill_null(True).sum().alias('reit_mismatch'),
    (pl.col('is_fund_diagnostic') != expected_fund).fill_null(True).sum().alias('fund_mismatch'),
    (
        pl.col('is_formation_eligible')
        & (
            pl.col('is_adr_diagnostic')
            | pl.col('is_reit_diagnostic')
            | pl.col('is_fund_diagnostic')
        )
    ).sum().alias('excluded_class_formation_rows'),
).collect(engine='streaming')

audit_row = flag_audit.row(0, named=True)
for column in FLAG_COLUMNS:
    assert audit_row[f'{column}_nulls'] == 0, f'Null flag: {column}'
for key in (
    'common_mismatch', 'regular_mismatch', 'exchange_mismatch',
    'formation_mismatch', 'adr_mismatch', 'reit_mismatch', 'fund_mismatch',
    'excluded_class_formation_rows',
):
    assert audit_row[key] == 0, f'{key}={audit_row[key]:,}'

assert audit_row['date_min'].date() >= START_DATE
assert audit_row['date_max'].date() <= END_DATE
assert audit_row['rows'] == output_meta.num_rows
assert audit_row['permnos'] == manifest['n_retained_permnos']

print('PASS all seven flags independently reconstructed')
print('PASS no ADR, REIT, or fund formation leakage')
display(flag_audit)

## 4. Exact identifier and full-path reconciliation with RAW

The expected identifier set is independently reconstructed from the immutable RAW panel using only seven projected columns. The compact output must contain exactly this set—neither more nor fewer identifiers. Its row count must then equal the number of dated RAW observations belonging to that set over 1990–2025.

This is the decisive check that formation filtering did not truncate later holding-period or delisting rows.

In [ ]:
raw_common_expr = (
    (normalized_code('SecurityType') == 'EQTY')
    & (normalized_code('SecuritySubType') == 'COM')
    & (normalized_code('ShareType') == 'NS')
    & (normalized_code('USIncFlg') == 'Y')
    & normalized_code('IssuerType').is_in(COMMON_ISSUER_TYPES)
).fill_null(False)

raw_lf = (
    pl.scan_parquet(SOURCE, low_memory=True, rechunk=False)
    .filter(pl.col('DlyCalDt').dt.date().is_between(START_DATE, END_DATE, closed='both'))
)

expected_permnos = (
    raw_lf
    .filter(raw_common_expr)
    .select('PERMNO')
    .unique()
    .collect(engine='streaming')
)

actual_permnos = output_lf.select('PERMNO').unique().collect(engine='streaming')

missing_permnos = expected_permnos.join(actual_permnos, on='PERMNO', how='anti')
unexpected_permnos = actual_permnos.join(expected_permnos, on='PERMNO', how='anti')
assert missing_permnos.is_empty(), f'Missing retained PERMNOs: {len(missing_permnos):,}'
assert unexpected_permnos.is_empty(), f'Unexpected retained PERMNOs: {len(unexpected_permnos):,}'

expected_path_rows = (
    raw_lf
    .select(['PERMNO', 'DlyCalDt'])
    .join(expected_permnos.lazy(), on='PERMNO', how='semi')
    .select(pl.len().alias('rows'))
    .collect(engine='streaming')['rows'][0]
)

assert output_meta.num_rows == expected_path_rows, (
    f'Full-path row mismatch: expected={expected_path_rows:,}, '
    f'actual={output_meta.num_rows:,}'
)

print(f'PASS exact ever-common PERMNO set: {len(actual_permnos):,}')
print(f'PASS exact retained daily paths    : {expected_path_rows:,} rows')

## 5. Cryptographic integrity

SHA-256 verification is sequential and memory-bounded. Output verification is enabled by default. Source verification is optional because it requires an additional complete 4 GiB read; enable it for the final archived release.

In [ ]:
def sha256_file(path: Path, block_size: int = 8 * 2**20) -> str:
    digest = sha256()
    with path.open('rb') as handle:
        while block := handle.read(block_size):
            digest.update(block)
    return digest.hexdigest()


if VERIFY_OUTPUT_SHA256:
    actual_output_sha256 = sha256_file(OUTPUT)
    assert actual_output_sha256 == manifest['output_sha256']
    print(f'PASS output SHA-256: {actual_output_sha256}')
else:
    print('SKIP output SHA-256')

if VERIFY_SOURCE_SHA256:
    expected_source_sha256 = manifest.get('source_sha256')
    assert expected_source_sha256, 'The manifest contains no source SHA-256.'
    actual_source_sha256 = sha256_file(SOURCE)
    assert actual_source_sha256 == expected_source_sha256
    print(f'PASS source SHA-256: {actual_source_sha256}')
else:
    print('SKIP source SHA-256 re-read; source metadata signature passed')

## 6. Final verdict

A complete run of this notebook must end with all assertions passing before `01_build_pit_big_small_caps.ipynb` is executed. The next validation layer is `tests/01_assert_pit_big_small_caps.ipynb`, which audits breakpoints, exact Top-100 membership, 60-month completeness, and holding-period construction.

In [ ]:
print('=' * 78)
print('ALL COMMON-STOCK SOURCE ASSERTIONS PASSED')
print('The compact source is admissible for PIT universe construction.')